# SpaceX Falcon 9 Interactive Dashboard with Plotly Dash

## Objective

The objective of this notebook is to build an interactive dashboard for exploring
SpaceX Falcon 9 launch and landing data using **Plotly Dash**.

The dashboard will:

1. Allow the user to select **all launch sites or an individual launch site**.
2. Display the **landing success distribution** using an interactive pie chart.
3. Allow the user to filter launches by **payload mass range**.
4. Display the relationship between **payload mass and landing outcome** using an interactive scatter plot.
5. Use Dash callbacks so that the visualizations update automatically when the user changes the controls.
6. Save presentation-ready interactive outputs for the capstone project.

## Notebook Workflow

This notebook follows the same structured workflow used in the previous capstone notebooks:

1. Import and verify required libraries.
2. Load and validate the wrangled Falcon 9 dataset.
3. Prepare dashboard-ready variables.
4. Build reusable visualization functions.
5. Create the Plotly Dash layout and interactive controls.
6. Connect controls to plots using callbacks.
7. Run and test the dashboard.
8. Save presentation-ready outputs.
9. Record a slide-ready methodology statement and GitHub placeholder.

## Optional: Install Required Packages

If `dash` or `plotly` is not installed in the current Python environment, uncomment
and run the following installation command once.

In [1]:
# Uncomment only if Dash or Plotly is not already installed.
%pip install dash plotly folium

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/8.9 MB ? eta -:--:--
   ----------------------------- ---------- 6.6/8.9 MB 35.9 MB/s eta 0:00:01
   ---------------------------------------- 8.9/8.9 MB 36.0 MB/s  0:00:00
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   -------------------------------------- - 9.4/9.7 MB 52.5 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 46.4 MB/s  0:00:00
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 46.6 MB/s  0:00:00

   -- -------------------------------------  1/15 [Werkzeug]
   -- -------------------------------------  1/15 [Werkzeug]
   -- -------------------------------------  1/15 [Werkzeug]
   ---------- -----------------------------  4/15 [pydantic-core]
   ------------- --------------------------  5/15 [plotly]
   ------------- -------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


## Import Required Libraries

Pandas and NumPy are used for data preparation. Plotly Express creates the
interactive charts, while Dash provides the dashboard layout, controls, and callbacks.

In [2]:
import os
import sys
import pandas as pd
import numpy as np
import plotly.express as px
import plotly
import dash

from dash import Dash, dcc, html, Input, Output

print("Python version:", sys.version.split()[0])
print("Plotly version:", plotly.__version__)
print("Dash version:", dash.__version__)
print("Libraries imported successfully.")

Python version: 3.13.1
Plotly version: 7.1.0
Dash version: 4.4.1
Libraries imported successfully.


## Load the Falcon 9 Dataset

The wrangled Falcon 9 dataset created earlier in the capstone workflow is used
as the data source for the interactive dashboard.

In [3]:
df = pd.read_csv("dataset_part_2.csv")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (90, 18)


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,Class
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857,0
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857,0
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857,0
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093,0
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857,0


## Verify Required Columns

The dashboard requires launch-site, payload-mass, landing-class, and booster-version
information. The following cell verifies that the required columns are available.

The notebook also recognizes the original IBM column names if an archived dashboard
dataset is used instead of the locally generated capstone dataset.

In [4]:
# Normalize common IBM dashboard column names to the names used in this capstone workflow.
column_aliases = {
    "Launch Site": "LaunchSite",
    "class": "Class",
    "Payload Mass (kg)": "PayloadMass",
    "Booster Version Category": "BoosterVersion"
}

df = df.rename(columns={
    old: new for old, new in column_aliases.items()
    if old in df.columns and new not in df.columns
})

required_columns = ["LaunchSite", "PayloadMass", "Class", "BoosterVersion"]

for col in required_columns:
    print(f"{col}: {'FOUND' if col in df.columns else 'MISSING'}")

missing_required = [col for col in required_columns if col not in df.columns]

if missing_required:
    raise KeyError(
        "Required dashboard columns are missing: "
        + ", ".join(missing_required)
    )

LaunchSite: FOUND
PayloadMass: FOUND
Class: FOUND
BoosterVersion: FOUND


## Prepare Dashboard Data

Rows missing information required by the dashboard are removed. A descriptive
landing-outcome label is then created from the binary `Class` variable.

- `Class = 1` → Successful Landing
- `Class = 0` → Unsuccessful Landing

In [5]:
dashboard_df = df[
    ["LaunchSite", "PayloadMass", "Class", "BoosterVersion"]
].copy()

dashboard_df["PayloadMass"] = pd.to_numeric(
    dashboard_df["PayloadMass"],
    errors="coerce"
)

dashboard_df["Class"] = pd.to_numeric(
    dashboard_df["Class"],
    errors="coerce"
)

dashboard_df = dashboard_df.dropna(
    subset=["LaunchSite", "PayloadMass", "Class", "BoosterVersion"]
).reset_index(drop=True)

dashboard_df["Class"] = dashboard_df["Class"].astype(int)

dashboard_df["LandingOutcome"] = dashboard_df["Class"].map({
    1: "Successful Landing",
    0: "Unsuccessful Landing"
})

print("Dashboard-ready records:", len(dashboard_df))
display(dashboard_df.head())

Dashboard-ready records: 90


,LaunchSite,PayloadMass,Class,BoosterVersion,LandingOutcome
0,CCAFS SLC 40,6104.959412,0,Falcon 9,Unsuccessful Landing
1,CCAFS SLC 40,525.000000,0,Falcon 9,Unsuccessful Landing
2,CCAFS SLC 40,677.000000,0,Falcon 9,Unsuccessful Landing
3,VAFB SLC 4E,500.000000,0,Falcon 9,Unsuccessful Landing
4,CCAFS SLC 40,3170.000000,0,Falcon 9,Unsuccessful Landing


## Review Dashboard Variables

Before building the dashboard, review the available launch sites, payload range,
landing outcomes, and booster versions.

In [6]:
print("Launch sites:")
print(sorted(dashboard_df["LaunchSite"].unique()))

print("\nPayload mass range:")
print(
    f"{dashboard_df['PayloadMass'].min():.2f} kg "
    f"to {dashboard_df['PayloadMass'].max():.2f} kg"
)

print("\nLanding outcomes:")
print(dashboard_df["LandingOutcome"].value_counts())

print("\nBooster versions:")
print(sorted(dashboard_df["BoosterVersion"].astype(str).unique()))

Launch sites:
['CCAFS SLC 40', 'KSC LC 39A', 'VAFB SLC 4E']

Payload mass range:
350.00 kg to 15600.00 kg

Landing outcomes:
LandingOutcome
Successful Landing      60
Unsuccessful Landing    30
Name: count, dtype: int64

Booster versions:
['Falcon 9']


# Task 1 – Prepare the Interactive Dashboard Controls

Two user controls are created:

1. **Launch-Site Dropdown** – allows the user to view all launch sites or one selected site.
2. **Payload-Mass Range Slider** – allows the user to restrict the analysis to a selected payload range.

These controls will later be connected to the dashboard charts using Dash callbacks.

In [7]:
site_options = [
    {"label": "All Sites", "value": "ALL"}
] + [
    {"label": site, "value": site}
    for site in sorted(dashboard_df["LaunchSite"].unique())
]

payload_min = int(np.floor(dashboard_df["PayloadMass"].min()))
payload_max = int(np.ceil(dashboard_df["PayloadMass"].max()))

mark_values = np.linspace(payload_min, payload_max, 5)
payload_marks = {
    int(round(value)): f"{int(round(value)):,}"
    for value in mark_values
}

print("Number of launch-site options:", len(site_options))
print("Payload slider range:", payload_min, "to", payload_max, "kg")
print("Slider marks:", payload_marks)

Number of launch-site options: 4
Payload slider range: 350 to 15600 kg
Slider marks: {350: '350', 4162: '4,162', 7975: '7,975', 11788: '11,788', 15600: '15,600'}


# Task 2 – Create the Landing-Success Pie Chart

The pie chart summarizes the proportion of successful and unsuccessful Falcon 9
first-stage landings.

When **All Sites** is selected, the chart summarizes the complete dataset. When
an individual launch site is selected, the chart updates to show only that site's
landing outcomes.

In [8]:
def create_success_pie(selected_site="ALL"):
    if selected_site == "ALL":
        filtered_df = dashboard_df.copy()
        title = "Landing Outcomes for All Launch Sites"
    else:
        filtered_df = dashboard_df[
            dashboard_df["LaunchSite"] == selected_site
        ].copy()
        title = f"Landing Outcomes for {selected_site}"

    outcome_counts = (
        filtered_df["LandingOutcome"]
        .value_counts()
        .rename_axis("LandingOutcome")
        .reset_index(name="Count")
    )

    fig = px.pie(
        outcome_counts,
        values="Count",
        names="LandingOutcome",
        title=title,
        hole=0.25
    )

    fig.update_traces(
        textposition="inside",
        textinfo="percent+label"
    )

    fig.update_layout(
        legend_title_text="Landing Outcome"
    )

    return fig


create_success_pie("ALL").show()

### Task 2 Observation

The pie chart provides a direct comparison of successful and unsuccessful
first-stage landing outcomes. The launch-site selector allows the same comparison
to be repeated for an individual facility without changing the underlying code.

# Task 3 – Create the Payload vs. Landing-Outcome Scatter Plot

The scatter plot examines the relationship between **payload mass** and
**landing outcome**.

The plot is filtered by both:

- the selected launch site, and
- the payload-mass range chosen by the user.

Booster version is used as the color grouping so that different vehicle
configurations can be compared within the same interactive visualization.

In [9]:
def create_payload_scatter(selected_site="ALL", payload_range=None):
    if payload_range is None:
        payload_range = [payload_min, payload_max]

    low, high = payload_range

    filtered_df = dashboard_df[
        dashboard_df["PayloadMass"].between(low, high)
    ].copy()

    if selected_site != "ALL":
        filtered_df = filtered_df[
            filtered_df["LaunchSite"] == selected_site
        ]

    if selected_site == "ALL":
        title = (
            f"Payload Mass vs. Landing Outcome – All Sites "
            f"({low:,}–{high:,} kg)"
        )
    else:
        title = (
            f"Payload Mass vs. Landing Outcome – {selected_site} "
            f"({low:,}–{high:,} kg)"
        )

    fig = px.scatter(
        filtered_df,
        x="PayloadMass",
        y="Class",
        color="BoosterVersion",
        hover_data=[
            "LaunchSite",
            "LandingOutcome",
            "PayloadMass",
            "BoosterVersion"
        ],
        labels={
            "PayloadMass": "Payload Mass (kg)",
            "Class": "Landing Outcome",
            "BoosterVersion": "Booster Version"
        },
        title=title
    )

    fig.update_yaxes(
        tickmode="array",
        tickvals=[0, 1],
        ticktext=["Unsuccessful", "Successful"]
    )

    fig.update_layout(
        legend_title_text="Booster Version"
    )

    return fig


create_payload_scatter(
    selected_site="ALL",
    payload_range=[payload_min, payload_max]
).show()

### Task 3 Observation

The scatter plot allows landing outcomes to be examined across the payload-mass
range while also distinguishing booster versions. Combining the payload slider
with the launch-site dropdown enables targeted exploration of specific operating
conditions.

# Task 4 – Build the Plotly Dash Application

The Dash application combines the launch-site dropdown, payload-range slider,
landing-success pie chart, and payload scatter plot into one interactive dashboard.

The page layout is organized so that the controls appear above the two
visualizations.

In [10]:
app = Dash(__name__)

app.layout = html.Div(
    [
        html.H1(
            "SpaceX Falcon 9 Launch Dashboard",
            style={
                "textAlign": "center",
                "marginBottom": "10px"
            }
        ),

        html.P(
            "Explore landing success by launch site and payload mass.",
            style={
                "textAlign": "center",
                "marginBottom": "25px"
            }
        ),

        html.Div(
            [
                html.Label(
                    "Select Launch Site:",
                    style={"fontWeight": "bold"}
                ),

                dcc.Dropdown(
                    id="site-dropdown",
                    options=site_options,
                    value="ALL",
                    clearable=False,
                    style={"marginBottom": "20px"}
                ),

                html.Label(
                    "Select Payload Mass Range (kg):",
                    style={"fontWeight": "bold"}
                ),

                dcc.RangeSlider(
                    id="payload-slider",
                    min=payload_min,
                    max=payload_max,
                    step=100,
                    marks=payload_marks,
                    value=[payload_min, payload_max],
                    tooltip={
                        "placement": "bottom",
                        "always_visible": False
                    }
                )
            ],
            style={
                "maxWidth": "1000px",
                "margin": "0 auto 30px auto",
                "padding": "0 20px"
            }
        ),

        html.Div(
            [
                dcc.Graph(id="success-pie-chart"),
                dcc.Graph(id="payload-scatter-chart")
            ],
            style={
                "maxWidth": "1200px",
                "margin": "0 auto"
            }
        )
    ]
)

print("Dash layout created successfully.")

Dash layout created successfully.


# Task 5 – Connect the Dashboard Controls with Callbacks

Dash callbacks make the dashboard interactive.

Whenever the user changes either the launch-site dropdown or the payload-range
slider, both charts are regenerated using the current selections.

In [11]:
@app.callback(
    Output("success-pie-chart", "figure"),
    Output("payload-scatter-chart", "figure"),
    Input("site-dropdown", "value"),
    Input("payload-slider", "value")
)
def update_dashboard(selected_site, payload_range):
    pie_figure = create_success_pie(selected_site)

    scatter_figure = create_payload_scatter(
        selected_site,
        payload_range
    )

    return pie_figure, scatter_figure


print("Dashboard callback registered successfully.")

Dashboard callback registered successfully.


# Task 6 – Run and Test the Interactive Dashboard

Run the Dash application below.

### Expected interactions

1. Change the **launch-site dropdown** and verify that both charts update.
2. Move either end of the **payload-range slider** and verify that the scatter plot updates.
3. Hover over chart elements to inspect detailed mission information.
4. Confirm that the pie chart correctly summarizes landing outcomes for the selected site.

> If the inline dashboard does not display in your Jupyter environment, change
> `jupyter_mode="inline"` to `jupyter_mode="external"` and open the displayed URL.

In [17]:
# Run the dashboard inside Jupyter.
# Stop the running app before starting another instance on the same port.

app.run(
    jupyter_mode="external", 
    debug=False,
    port=8050
)

Dash app running on http://127.0.0.1:8050/


# Task 7 – Create Presentation-Ready Dashboard Outputs

Although the Dash application itself is interactive, default versions of the two
main figures are also saved as standalone HTML files. These files are useful for
review, archiving, and preparing the final capstone presentation.

In [13]:
default_pie = create_success_pie("ALL")

default_scatter = create_payload_scatter(
    selected_site="ALL",
    payload_range=[payload_min, payload_max]
)

default_pie.write_html(
    "dash_01_all_sites_success_pie.html"
)

default_scatter.write_html(
    "dash_02_payload_success_scatter.html"
)

print("Saved: dash_01_all_sites_success_pie.html")
print("Saved: dash_02_payload_success_scatter.html")

Saved: dash_01_all_sites_success_pie.html
Saved: dash_02_payload_success_scatter.html


## Create a Launch-Site Landing Summary

A compact launch-site summary is exported as a CSV file to support interpretation
of the dashboard and preparation of the final presentation.

In [14]:
site_summary = (
    dashboard_df
    .groupby("LaunchSite")
    .agg(
        Total_Launches=("Class", "size"),
        Successful_Landings=("Class", "sum")
    )
    .reset_index()
)

site_summary["Unsuccessful_Landings"] = (
    site_summary["Total_Launches"]
    - site_summary["Successful_Landings"]
)

site_summary["Success_Rate_Percent"] = (
    site_summary["Successful_Landings"]
    / site_summary["Total_Launches"]
    * 100
).round(2)

display(site_summary)

site_summary.to_csv(
    "dash_03_launch_site_summary.csv",
    index=False
)

print("Saved: dash_03_launch_site_summary.csv")

,LaunchSite,Total_Launches,Successful_Landings,Unsuccessful_Landings,Success_Rate_Percent
0,CCAFS SLC 40,55,33,22,60.00
1,KSC LC 39A,22,17,5,77.27
2,VAFB SLC 4E,13,10,3,76.92


Saved: dash_03_launch_site_summary.csv


# Capstone Presentation Outputs

## Interactive Dashboard Summary

This notebook used **Plotly Dash** to create an interactive dashboard for
exploring Falcon 9 launch and landing performance.

The dashboard includes:

- A **launch-site dropdown** for selecting all sites or one individual facility.
- A **payload-mass range slider** for filtering launches by payload.
- An interactive **pie chart** showing successful and unsuccessful landing outcomes.
- An interactive **scatter plot** showing the relationship between payload mass
  and landing outcome.
- **Booster-version color grouping** to compare vehicle configurations.
- Dash **callbacks** that automatically update both visualizations when the user
  changes the dashboard controls.

## Slide-Ready Methodology Statement

**Interactive visual analytics were performed using Plotly Dash.** A launch-site
dropdown and payload-mass range slider were used as dashboard inputs. A landing
outcome pie chart summarized first-stage landing success for all sites or a
selected launch site, while an interactive scatter plot examined the relationship
between payload mass and landing outcome by booster version. Dash callbacks linked
the user controls to the visualizations so that the charts updated automatically
as selections changed.

## Saved Dashboard Outputs

1. `dash_01_all_sites_success_pie.html` – Interactive landing-outcome pie chart
2. `dash_02_payload_success_scatter.html` – Interactive payload-versus-outcome scatter plot
3. `dash_03_launch_site_summary.csv` – Launch-site landing summary


## Verify Saved Outputs

The following cell confirms that all presentation-ready dashboard outputs were
created successfully.

In [15]:
output_files = [
    "dash_01_all_sites_success_pie.html",
    "dash_02_payload_success_scatter.html",
    "dash_03_launch_site_summary.csv"
]

for file in output_files:
    print(
        f"{file}:",
        "SAVED" if os.path.exists(file) else "MISSING"
    )

dash_01_all_sites_success_pie.html: SAVED
dash_02_payload_success_scatter.html: SAVED
dash_03_launch_site_summary.csv: SAVED


## Final Observation

The Plotly Dash dashboard provides an interactive way to explore Falcon 9 landing
performance across launch sites and payload conditions. The combination of a
site selector, payload-range filter, landing-outcome pie chart, and payload
scatter plot allows the user to move from an overall view of launch performance
to more targeted comparisons of specific sites and operating ranges.

The dashboard therefore complements the static EDA and SQL analyses by allowing
stakeholders to explore the Falcon 9 dataset interactively.

# Author

<a href="https://www.linkedin.com/in/aung-myat-phd-ceng-mimeche-8810915b/">Aung Myat</a>

IBM Data Science Professional Certificate Capstone
Reproduced and completed in 2026
